# Image Classification with PyTorch and Gradio

This notebook demonstrates how to build a web-based image classification application using a pretrained ResNet-18 model and the Gradio library.

### Step 1: Setting up the image classification model

First, we will load a pretrained ResNet-18 model from PyTorch Hub. ResNet-18 is a popular convolutional neural network that has been trained on the ImageNet dataset.

In [ ]:
import torch

# Load pretrained ResNet-18 model
model = torch.hub.load('pytorch/vision:v0.6.0', 'resnet18', pretrained=True).eval()

### Step 2: Defining a predict function

Next, we define a function that takes an image as input and returns the predicted classes and their confidence scores. We'll also need to download the ImageNet labels to map the model's numerical output to human-readable names.

In [ ]:
import requests
from torchvision import transforms
from PIL import Image

# Download human-readable labels for ImageNet
response = requests.get("https://git.io/JJkYN")
labels = [l.strip() for l in response.text.split("\n") if l.strip()]

# Define image preprocessing (Normalization is crucial for ResNet)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

def predict(inp):
    # Preprocess image
    inp = transform(inp).unsqueeze(0)

    # Ensure model runs in inference mode (no gradient calculation)
    with torch.no_grad():
        prediction = torch.nn.functional.softmax(model(inp)[0], dim=0)

    # Map predictions to labels and return as a dictionary
    confidences = {
        labels[i]: float(prediction[i]) 
        for i in range(len(labels))
    }
    
    return confidences

### Step 3: Creating a Gradio interface

Finally, we use Gradio to create an interactive UI. The user can upload an image, and the model will display the top 3 predicted labels.

In [ ]:
import gradio as gr

# Create and launch the Gradio interface
interface = gr.Interface(
    fn=predict, 
    inputs=gr.Image(type="pil"), 
    outputs=gr.Label(num_top_classes=3),
    examples=["lion.jpg", "cheetah.jpg"]  # Ensure these files exist in the same directory or provide full paths
)

interface.launch()